### 1. Data Loading and Initial Split

In [ ]:
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, pipeline, AutoModelForSeq2SeqLM
from sklearn.metrics import accuracy_score, f1_score
import nltk
from nltk.corpus import wordnet
import random
from datasets import Dataset, concatenate_datasets
import pandas as pd

In [ ]:

# Load dataset
dataset = load_dataset("pubmed_qa", "pqa_labeled")
data_original = dataset["train"]

# Initial split into original train and test sets
split_dataset = data_original.train_test_split(test_size=0.2, seed=42)
main_train_data = split_dataset["train"]
main_test_data = split_dataset["test"]



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

### Data Augmentation Techniques

#### 1. Paraphrasing Augmentation

In [ ]:
paraphrase_tokenizer = AutoTokenizer.from_pretrained("ramsrigouthamg/t5_paraphraser")
paraphrase_model = AutoModelForSeq2SeqLM.from_pretrained("ramsrigouthamg/t5_paraphraser")

def generate_paraphrases(text, num_return_sequences=1):
    prompt = "paraphrase: " + text + " </s>"

    inputs = paraphrase_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=64
    )

    outputs = paraphrase_model.generate(
        **inputs,
        max_length=64,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7
    )


    results = [
        paraphrase_tokenizer.decode(o, skip_special_tokens=True)
        for o in outputs
    ]

    # remove duplicates & identical outputs
    results = list(set([
        r for r in results if r.lower() != text.lower()
    ]))

    return results

# The generation of paraphrased_data_list will be moved to a new cell to implement
# the combined augmentation strategy.

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


## original + paraphrased(ques + context)

In [ ]:
from tqdm import tqdm
from datasets import Dataset
import copy
import re
import random
from nltk.tokenize import sent_tokenize
import nltk
nltk.download('punkt_tab')


# -------- CONFIG --------
WH_WORDS = ("who", "what", "where", "when", "why", "which", "whose", "how")


# -------- HELPERS --------
def is_valid_question(q):
    q = q.strip().lower()
    return not any(q.startswith(w) for w in WH_WORDS)


def normalize_text(text):
    return re.sub(r"\s+", " ", text.strip().lower())


def get_hashable(item):
    question = normalize_text(item["question"])
    context = normalize_text(" ".join(item["context"].get("contexts", [])))
    return (question, context)


def split_sentences(text):
    """Robust sentence splitter"""
    return sent_tokenize(text)

def augment_with_paraphrase(item, num_paraphrases_per_item=1):
    augmented_samples = []

    original_question = item["question"]
    original_context_dict = item["context"]
    original_final_decision = item["final_decision"]


    # =========================
    # 1. Paraphrase QUESTION → pick ONE
    # =========================
    paraphrased_questions = generate_paraphrases(original_question, 3)

    if not paraphrased_questions:
        return []

    pq = None
    original_q_norm = normalize_text(original_question)

    for q in paraphrased_questions:
        if is_valid_question(q) and normalize_text(q) != original_q_norm:
          pq = q
          break

    if pq is None:
        return []
    # =========================
    # 2. Paraphrase CONTEXT (sentence-wise, SAME structure)
    # =========================
    new_context_dict = copy.deepcopy(original_context_dict)
    paraphrased_blocks = []



    for block in original_context_dict["contexts"]:
        sentences = split_sentences(block)
        paraphrased_sentences = []
        i=1
        for sentence in sentences:
            i+=1
            paras = generate_paraphrases(sentence, 3)

            if paras:
                chosen = random.choice(paras)
            else:
                chosen = sentence

            # ensure punctuation
            chosen = chosen.strip()
            if not chosen.endswith((".", "?", "!")):
                chosen += "."
            paraphrased_sentences.append(chosen)

    block_text = " ".join(paraphrased_sentences)
    paraphrased_blocks.append(block_text)

    new_context_dict["contexts"] = paraphrased_blocks

    # =========================
    # FINAL SINGLE AUGMENTED SAMPLE
    # =========================
    augmented_samples.append({
        "pubid": None,
        "question": pq,
        "context": new_context_dict,
        "long_answer": None,
        "final_decision": original_final_decision
    })
    return augmented_samples

paraphrased_data_list = []
for item in tqdm(main_train_data, desc="Generating paraphrased data"):
    # Add original item first
    paraphrased_data_list.append({
        "pubid": item["pubid"],
        "question": item["question"],
        "context": item["context"],
        "long_answer": item["long_answer"],
        "final_decision": item["final_decision"]
    })
    # Add augmented items
    paraphrased_data_list.extend(augment_with_paraphrase(item, num_paraphrases_per_item=1))

paraphrased_dataset = Dataset.from_list(paraphrased_data_list)


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
Generating paraphrased data: 100%|██████████| 800/800 [1:58:51<00:00,  8.91s/it]

Paraphrased data samples (including original): 1428


In [ ]:
from google.colab import files
paraphrased_dataset.to_csv("paraphrased_dataset.csv", index=False)
files.download("paraphrased_dataset.csv")

Creating CSV from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 Preprocessing and Tokenization Pipeline

In [ ]:
def flatten_context(context):
    if isinstance(context, dict):
        parts = []
        for value in context.values():
            if isinstance(value, list):
                parts.append(" ".join(value))
            elif isinstance(value, str):
                parts.append(value)
        return " ".join(parts)
    return context

def preprocess(example):
    return {
        "question": example["question"],
        "context": flatten_context(example["context"]),
        "label": example["final_decision"]
    }

aug_train_data = aug_train_data.map(preprocess)
main_test_data = main_test_data.map(preprocess)

label_map = {"yes": 0, "no": 1, "maybe": 2}

def encode_label(example):
    example["label"] = label_map[example["label"]]
    return example

aug_train_data = aug_train_data.map(encode_label)
main_test_data = main_test_data.map(encode_label)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.1"
)
# tokenizer = AutoTokenizer.from_pretrained(
#     "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"
# )



def tokenize(example):
    return tokenizer(
        example["question"],
        example["context"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

aug_train_data = aug_train_data.map(tokenize, batched=True)
main_test_data = main_test_data.map(tokenize, batched=True)

aug_train_data.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "token_type_ids", "label"]
)

main_test_data.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "token_type_ids", "label"]
)

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Preprocessed and tokenized augmented training data:
Dataset({
    features: ['pubid', 'question', 'context', 'long_answer', 'final_decision', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 800
})
Preprocessed and tokenized original test data:
Dataset({
    features: ['pubid', 'question', 'context', 'long_answer', 'final_decision', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 200
})


 Training and Evaluation Loop

In [ ]:
# Model
model = AutoModelForSequenceClassification.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.1",
    num_labels=3
)
# model = AutoModelForSequenceClassification.from_pretrained(
#     "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract",
#     num_labels=3
# )

# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro")
    }

# Training args
training_args = TrainingArguments(
    output_dir="./results_combined_augmentation",
    eval_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    logging_dir='./logs_combined_augmentation',
    logging_steps=10,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=aug_train_data,
    eval_dataset=main_test_data,
    compute_metrics=compute_metrics
)

print("Starting training...")
trainer.train()

print("Starting evaluation...")
results = trainer.evaluate()
print("Evaluation results:")
print(results)

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne

Starting training...


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.297099,0.496254,0.835000,0.788311
2,0.217819,0.622475,0.830000,0.786808
3,0.156536,0.509726,0.860000,0.809228


... (2 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Starting evaluation...


Evaluation results:
{'eval_loss': 0.5097258687019348, 'eval_accuracy': 0.86, 'eval_f1': 0.8092277010296572, 'eval_runtime': 1.3396, 'eval_samples_per_second': 149.294, 'eval_steps_per_second': 18.662, 'epoch': 3.0}


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

print("Generating predictions...")

predictions = trainer.predict(main_test_data)

logits = predictions.predictions
y_pred = np.argmax(logits, axis=1)
y_true = predictions.label_ids

labels = ["yes", "no", "maybe"]

print("\nConfusion Matrix:")
cm = confusion_matrix(y_true, y_pred)
print(cm)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=labels))

Generating predictions...



Confusion Matrix:
[[88  9  6]
 [19 50  3]
 [ 3  2 20]]

Classification Report:
              precision    recall  f1-score   support

         yes       0.80      0.85      0.83       103
          no       0.82      0.69      0.75        72
       maybe       0.69      0.80      0.74        25

    accuracy                           0.79       200
   macro avg       0.77      0.78      0.77       200
weighted avg       0.79      0.79      0.79       200



### 2. RAG Based Context Retrieval + Augmentation



In [ ]:
!pip install sentence-transformers faiss-cpu datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 117.4 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset

dataset = load_dataset("pubmed_qa", "pqa_labeled")
unlabeled_dataset = load_dataset("pubmed_qa", "pqa_unlabeled")

data_original = dataset["train"]

# Initial split into original train and test sets
split_dataset = data_original.train_test_split(test_size=0.2, seed=42)

train_data = split_dataset["train"]
test_data = split_dataset["test"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

pqa_unlabeled/train-00000-of-00001.parqu(…):   0%|          | 0.00/66.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/61249 [00:00<?, ? examples/s]

In [ ]:
def flatten_context(context):
    if isinstance(context, dict):
        parts = []
        for value in context.values():
            if isinstance(value, list):
                parts.append(" ".join(value))
            elif isinstance(value, str):
                parts.append(value)
        return " ".join(parts)
    return context

def preprocess(example):
    return {
        "question": example["question"],
        "context": flatten_context(example["context"]),
        "label": example["final_decision"]
    }

# aug_train_data = aug_train_data.map(preprocess)
main_test_data = test_data.map(preprocess)

label_map = {"yes": 0, "no": 1, "maybe": 2}

def encode_label(example):
    example["label"] = label_map[example["label"]]
    return example

# aug_train_data = aug_train_data.map(encode_label)
main_test_data = main_test_data.map(encode_label)

# Tokenizer
# tokenizer = AutoTokenizer.from_pretrained(
#     "dmis-lab/biobert-base-cased-v1.1"
# )
# print("example: ", aug_train_data["question"])
tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"
)


def tokenize(example):
    return tokenizer(
        example["question"],
        example["context"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

# aug_train_data = aug_train_data.map(tokenize, batched=True)
main_test_data = main_test_data.map(tokenize, batched=True)

# aug_train_data.set_format(
#     type="torch",
#     columns=["input_ids", "attention_mask", "token_type_ids", "label"]
# )

main_test_data.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "token_type_ids", "label"]
)


print("Preprocessed and tokenized original test data:")
print(main_test_data)


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

NameError: name 'AutoTokenizer' is not defined

In [ ]:

def flatten_context(context):
    if isinstance(context, dict):
        all_parts = []

        for value in context.values():
            if isinstance(value, list):
                all_parts.append(" ".join(value))  # join sentences
            elif isinstance(value, str):
                all_parts.append(value)

        return " ".join(all_parts)

    return context
    # Ensure context exists and is not empty
    # context_text = flatten_context(example["context"])

def clean_labeled(example):
    return {
        "question": example["question"],
        "context": flatten_context(example["context"]),
        "label": example["final_decision"]
    }

def clean_unlabeled(example):
    return {
        "context": flatten_context(example["context"])
    }

train_data = train_data.map(clean_labeled)
print(train_data)
train_data = train_data.remove_columns(
    [col for col in train_data.column_names if col not in ["question", "context", "label"]]
)

unlabeled_dataset = unlabeled_dataset["train"].map(clean_unlabeled)
unlabeled_dataset = unlabeled_dataset.remove_columns(
    [col for col in unlabeled_dataset.column_names if col != "context"]
)

Dataset({
    features: ['pubid', 'question', 'context', 'long_answer', 'final_decision', 'label'],
    num_rows: 800
})


In [ ]:
corpus = unlabeled_dataset["context"]
print("Corpus size:", len(corpus))  # ~61k

Corpus size: 61249


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")

corpus_embeddings = embedder.encode(
    corpus,
    convert_to_numpy=True,
    show_progress_bar=True,
    batch_size=64
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/958 [00:00<?, ?it/s]

In [ ]:
import faiss

dimension = corpus_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(corpus_embeddings)

print("Index size:", index.ntotal)

Index size: 61249


In [ ]:
def retrieve_context(question, k=2):

    query_embedding = embedder.encode([question], convert_to_numpy=True)

    distances, indices = index.search(query_embedding, k)

    # Convert numpy.int64 indices to Python int for proper dataset indexing
    retrieved = [corpus[int(i)] for i in indices[0]]

    return retrieved

In [ ]:
rag_data = []

for item in train_data:

    retrieved_contexts = retrieve_context(item["question"], k=2)

    combined_context = item["context"] + " " + " ".join(retrieved_contexts)

    # truncate to avoid token overflow
    combined_context = combined_context[:1000]

    rag_data.append({
        "question": item["question"],
        "context": combined_context,
        "label": item["label"]
    })

In [ ]:
from datasets import Dataset

rag_dataset = Dataset.from_list(rag_data)

print("RAG dataset size:", len(rag_dataset))

RAG dataset size: 800


In [ ]:
def preprocess(example):
        return {
        "question": example["question"],
        "context": flatten_context(example["context"]),
        "label": example["label"]
    }

rag_dataset = rag_dataset.map(preprocess)

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

In [ ]:
label_map = {"yes": 0, "no": 1, "maybe": 2}

def encode_label(example):
    example["label"] = label_map[example["label"]]
    return example

rag_dataset = rag_dataset.map(encode_label)

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoTokenizer

# Tokenizer
# tokenizer = AutoTokenizer.from_pretrained(
#     "dmis-lab/biobert-base-cased-v1.1"
# )
tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"
)



def tokenize(example):
        return tokenizer(
        example["question"],
        example["context"],
        truncation=True,
        padding="max_length",
        max_length=512
    )


rag_dataset = rag_dataset.map(tokenize, batched=True)
rag_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

In [ ]:
# Keeps only columns in df1 that also exist in df2
main_test_data = main_test_data.remove_columns(
    [col for col in main_test_data.column_names if col not in ['question', 'context', 'label', 'text', 'input_ids', 'token_type_ids', 'attention_mask']])


In [ ]:
# Model
# model = AutoModelForSequenceClassification.from_pretrained(
#     "dmis-lab/biobert-base-cased-v1.1",
#     num_labels=3
# )

model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract",
    num_labels=3
)

# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro")
    }

# Training args
training_args = TrainingArguments(
    output_dir="./results_combined_augmentation",
    eval_strategy="epoch",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    logging_dir='./logs_combined_augmentation',
    logging_steps=10,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=rag_dataset,
    eval_dataset=main_test_data,
    compute_metrics=compute_metrics
)

print("Starting training...")
trainer.train()

print("Starting evaluation...")
results = trainer.evaluate()
print("Evaluation results:")
print(results)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:

Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.994534,0.898393,0.595000,0.416895
2,0.863765,0.857227,0.610000,0.392691
3,0.573671,0.698291,0.775000,0.550938
4,0.400305,0.897063,0.745000,0.529647
5,0.249752,1.069277,0.740000,0.526020


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (2 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Starting evaluation...


Evaluation results:
{'eval_loss': 0.6992391347885132, 'eval_accuracy': 0.775, 'eval_f1': 0.550937950937951, 'eval_runtime': 1.3618, 'eval_samples_per_second': 146.863, 'eval_steps_per_second': 18.358, 'epoch': 5.0}


In [ ]:

from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

print("Generating predictions...")

predictions = trainer.predict(main_test_data)

logits = predictions.predictions
y_pred = np.argmax(logits, axis=1)
y_true = predictions.label_ids

labels = ["yes", "no", "maybe"]

print("\nConfusion Matrix:")
cm = confusion_matrix(y_true, y_pred)
print(cm)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=labels))

Generating predictions...



Confusion Matrix:
[[87 16  0]
 [ 4 68  0]
 [16  9  0]]

Classification Report:
              precision    recall  f1-score   support

         yes       0.81      0.84      0.83       103
          no       0.73      0.94      0.82        72
       maybe       0.00      0.00      0.00        25

    accuracy                           0.78       200
   macro avg       0.51      0.60      0.55       200
weighted avg       0.68      0.78      0.72       200



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# 3. Bio Word2vec Augmentation


In [ ]:
from google.colab import files

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score

# Assuming train_data and val_data DataFrames are already loaded from the CSV files
# and are available in the environment.
# If not, ensure these lines are uncommented or executed previously:
# import pandas as pd
# train_data = pd.read_csv('pubmedqa_aug_train.csv')
# val_data = pd.read_csv('pubmedqa_val.csv')

# Convert Pandas DataFrames to Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_data)
val_dataset = Dataset.from_pandas(val_data)

# 1. Preprocessing function
# def preprocess(example):
#     # Ensure context exists and is not empty, combine question and context
#     context_text = example["context"]
#     text = example["question"] + " " + context_text
#     label = example["label"]
#     return {"text": text, "label": label}

# Redefine flatten_context here too, as it's used by preprocess
def flatten_context(context):
    if isinstance(context, dict):
        parts = []
        for value in context.values():
            if isinstance(value, list):
                parts.append(" ".join(value))
            elif isinstance(value, str):
                parts.append(value)
        return " ".join(parts)
    return context


def preprocess(example):
    return {
        "question": example["question"],
        "context": flatten_context(example["context"]),
        "label": example["label"]
    }

# Apply preprocessing
train_dataset = train_dataset.map(preprocess)
val_dataset = val_dataset.map(preprocess)

# Encode labels
label_map = {"yes": 0, "no": 1, "maybe": 2}

def encode_label(example):
    # Check if the label is already an integer (e.g., from prior processing or CSV)
    if isinstance(example["label"], str):
        example["label"] = label_map[example["label"]]
    return example

train_dataset = train_dataset.map(encode_label)
val_dataset = val_dataset.map(encode_label)

# 2. Tokenization with BioBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.1"
)
# tokenizer = AutoTokenizer.from_pretrained(
#     "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"
# )

def tokenize(example):
    # Explicitly set max_length to a common value to ensure consistent tensor sizes
    # return tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)
        return tokenizer(
        example["question"],
        example["context"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

print("Preprocessed and tokenized training data:")
print(train_dataset)
print("Preprocessed and tokenized validation data:")
print(val_dataset)

# 3. Model, Training Arguments, and Trainer
model = AutoModelForSequenceClassification.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.1",
    num_labels=3 # 'yes', 'no', 'maybe'
)
# model = AutoModelForSequenceClassification.from_pretrained(
#     "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract",
#     num_labels=3
# )

# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro")
    }

# Training args
training_args = TrainingArguments(
    output_dir="./results_biobert_csv",
    eval_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    logging_dir='./logs_biobert_csv',
    logging_steps=10,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to=[] # Disable reporting to services like Weights & Biases
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("Starting training...")
trainer.train()

print("Starting evaluation...")
results = trainer.evaluate()
print("Evaluation results:")
print(results)


Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

... (7 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Starting evaluation...


Evaluation results:
{'eval_loss': 0.4708082675933838, 'eval_accuracy': 0.85, 'eval_f1': 0.8237004548408057, 'eval_runtime': 1.313, 'eval_samples_per_second': 152.327, 'eval_steps_per_second': 19.041, 'epoch': 3.0}


In [ ]:
# Download the specific BioWordVec binary file
!wget https://ftp.ncbi.nlm.nih.gov/pub/lu/Suppl/BioSentVec/BioWordVec_PubMed_MIMICIII_d200.vec.bin

--2026-04-16 19:11:11--  https://ftp.ncbi.nlm.nih.gov/pub/lu/Suppl/BioSentVec/BioWordVec_PubMed_MIMICIII_d200.vec.bin
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.12, 130.14.250.13, 2607:f220:41e:250::7, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.12|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13451441787 (13G) [application/octet-stream]
Saving to: ‘BioWordVec_PubMed_MIMICIII_d200.vec.bin’

BioWordVec_PubMed_M 100%[===================>]  12.53G  11.9MB/s    in 18m 10s 

2026-04-16 19:29:23 (11.8 MB/s) - ‘BioWordVec_PubMed_MIMICIII_d200.vec.bin’ saved [13451441787/13451441787]



In [ ]:
!pip install gensim

from gensim.models import KeyedVectors

# Load using Gensim (this also requires High-RAM)
w2v_model = KeyedVectors.load_word2vec_format('BioWordVec_PubMed_MIMICIII_d200.vec.bin', binary=True)

# To find replacements in Gensim:
# replacement = model.most_similar('hypotension', topn=1)[0][0]

# from gensim.models import FastText
# model = FastText.load_fasttext_format('BioWordVec_PubMed_MIMICIII_d200.bin')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 96.6 MB/s eta 0:00:00


In [ ]:
# from gensim.models import KeyedVectors
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
from google.colab import files
stopwordsfile = files.upload()

# To access the file later:
# import pandas as pd
# df = pd.read_csv('your_file_name.csv')

Saving stopwords.pkl to stopwords.pkl


In [ ]:
from pprint import pprint
import os, pickle, re, json, random, sys
from tqdm import tqdm


In [ ]:
from nltk.corpus import stopwords

sws = set(stopwords.words('english'))
with open('stopwords.pkl', 'rb') as f:
	stopwords = pickle.load(f)


In [ ]:
import re

def is_safe_token(token):
    # avoid numbers, short tokens, uppercase entities
    if len(token) <=3 :
        return False
    if re.search(r'\d', token):
        return False
    if token.isupper():
        return False
    if token in stopwords:
        return False
    return True

In [ ]:
import random

def substitute_words(text, model, max_replacements=5, similarity_threshold=0.90):
    parts = re.split(r'(\W+)', text)
    candidate_indices = [
        i for i, part in enumerate(parts)
        if re.fullmatch(r'\w+', part)
        and part in model.key_to_index
        and is_safe_token(part)
    ]

    if not candidate_indices:
        return text

    num_replacements = min(max_replacements, len(candidate_indices))
    indices_to_replace = random.sample(candidate_indices, num_replacements)

    for idx in indices_to_replace:
        word = parts[idx].lower()
        try:
            similar_words = model.most_similar(word, topn=20)
            filtered = [w for w, sim in similar_words if sim >= similarity_threshold]
            if filtered:
                parts[idx] = random.choice(filtered)
        except KeyError:
            continue

    return ''.join(parts)


In [ ]:
def flatten_context(context):
    if isinstance(context, dict):
        all_parts = []

        for value in context.values():
            if isinstance(value, list):
                all_parts.append(" ".join(value))  # join sentences
            elif isinstance(value, str):
                all_parts.append(value)

        return " ".join(all_parts)

    return context

In [ ]:

def augment_context(context, model):
    if isinstance(context, dict) and "contexts" in context:
        new_context = context.copy()
        new_context["contexts"] = [
            substitute_words(sent, model)
            for sent in context["contexts"]
        ]
        return new_context

    # fallback for other formats
    context_text = flatten_context(context)
    augmented_text = substitute_words(context_text, model)
    return augmented_text

### Test Augmentation Functionality

In [ ]:
def build_dataset(data, w2v_model, augment=False, max_samples=8000):
    questions = []
    contexts = []
    labels = []

    label_map = {"yes": 0, "no": 1, "maybe": 2}

    for i, sample in enumerate(tqdm(data, desc="Building dataset")):
        context = sample["context"]
        question = sample["question"]
        label = sample["final_decision"]

        if label not in label_map:
            continue

        if augment:
            context = augment_context(context, w2v_model)

        context_text = flatten_context(context)

        questions.append(question)
        contexts.append(context_text)
        labels.append(label_map[label])

        if augment and len(labels) >= max_samples:
            break

    return questions, contexts, labels

In [ ]:
from datasets import load_dataset

dataset = load_dataset("pubmed_qa", "pqa_labeled")
# unlabeled_dataset = load_dataset("pubmed_qa", "pqa_unlabeled")

data_original = dataset["train"]

# Initial split into original train and test sets
split_dataset = data_original.train_test_split(test_size=0.2, seed=42)

train_data = split_dataset["train"]
val_data = split_dataset["test"]

In [ ]:
# Original
train_questions, train_contexts, train_labels = build_dataset(train_data, w2v_model, augment=False)

# Augmented (small!)
aug_questions, aug_contexts, aug_labels = build_dataset(train_data, w2v_model, augment=True, max_samples=8000)

# Combine
train_questions += aug_questions
train_contexts += aug_contexts
train_labels += aug_labels

# Validation
val_questions, val_contexts, val_labels = build_dataset(val_data, w2v_model, augment=False)

Building dataset:   0%|          | 0/800 [00:00<?, ?it/s]

parts:  ['Figures', ' ', 'from', ' ', 'the', ' ', 'British', ' ', 'Defence', ' ', 'Dental', ' ', 'Services', ' ', 'reveal', ' ', 'that', ' ', 'serving', ' ', 'personnel', ' ', 'in', ' ', 'the', ' ', 'British', ' ', 'Army', ' ', 'have', ' ', 'a', ' ', 'persistently', ' ', 'lower', ' ', 'level', ' ', 'of', ' ', 'dental', ' ', 'fitness', ' ', 'than', ' ', 'those', ' ', 'in', ' ', 'the', ' ', 'Royal', ' ', 'Navy', ' ', 'or', ' ', 'the', ' ', 'Royal', ' ', 'Air', ' ', 'Force', '. ', 'No', ' ', 'research', ' ', 'had', ' ', 'been', ' ', 'undertaken', ' ', 'to', ' ', 'ascertain', ' ', 'if', ' ', 'this', ' ', 'reflects', ' ', 'the', ' ', 'oral', ' ', 'health', ' ', 'of', ' ', 'recruits', ' ', 'joining', ' ', 'each', ' ', 'Service', '. ', 'This', ' ', 'study', ' ', 'aimed', ' ', 'to', ' ', 'pilot', ' ', 'a', ' ', 'process', ' ', 'for', ' ', 'collecting', ' ', 'dental', ' ', 'and', ' ', 'sociodemographic', ' ', 'data', ' ', 'from', ' ', 'new', ' ', 'recruits', ' ', 'to', ' ', 'each', ' ', 'Servic

Building dataset:   0%|          | 1/800 [00:16<3:46:12, 16.99s/it]

parts:  ['To', ' ', 'assess', ' ', 'pain', '-', 'related', ' ', 'attentional', ' ', 'biases', ' ', 'among', ' ', 'individuals', ' ', 'with', ' ', 'episodic', ' ', 'migraine', '.', '']
parts:  ['Prior', ' ', 'studies', ' ', 'have', ' ', 'examined', ' ', 'whether', ' ', 'chronic', ' ', 'pain', ' ', 'patients', ' ', 'selectively', ' ', 'attend', ' ', 'to', ' ', 'pain', '-', 'related', ' ', 'stimuli', ' ', 'in', ' ', 'the', ' ', 'environment', ', ', 'but', ' ', 'these', ' ', 'studies', ' ', 'have', ' ', 'produced', ' ', 'largely', ' ', 'mixed', ' ', 'findings', ' ', 'and', ' ', 'focused', ' ', 'primarily', ' ', 'on', ' ', 'patients', ' ', 'with', ' ', 'chronic', ' ', 'musculoskeletal', ' ', 'pain', '. ', 'Limited', ' ', 'research', ' ', 'has', ' ', 'implicated', ' ', 'attentional', ' ', 'biases', ' ', 'among', ' ', 'chronic', ' ', 'headache', ' ', 'patients', ', ', 'but', ' ', 'no', ' ', 'studies', ' ', 'have', ' ', 'been', ' ', 'conducted', ' ', 'among', ' ', 'episodic', ' ', 'migraineurs

Building dataset:   0%|          | 2/800 [00:24<2:34:42, 11.63s/it]

parts:  ['This', ' ', 'study', ' ', 'was', ' ', 'designed', ' ', 'to', ' ', 'determine', ' ', 'whether', ' ', 'preclerkship', ' ', 'performance', ' ', 'examinations', ' ', 'could', ' ', 'accurately', ' ', 'identify', ' ', 'medical', ' ', 'students', ' ', 'at', ' ', 'risk', ' ', 'for', ' ', 'failing', ' ', 'a', ' ', 'senior', ' ', 'clinical', ' ', 'performance', ' ', 'examination', ' (', 'CPE', ').', '']
parts:  ['This', ' ', 'study', ' ', 'used', ' ', 'a', ' ', 'retrospective', ' ', 'case', '-', 'control', ', ', 'multiyear', ' ', 'design', ', ', 'with', ' ', 'contingency', ' ', 'table', ' ', 'analyses', ', ', 'to', ' ', 'examine', ' ', 'the', ' ', 'performance', ' ', 'of', ' ', '412', ' ', 'students', ' ', 'in', ' ', 'the', ' ', 'classes', ' ', 'of', ' ', '2005', ' ', 'to', ' ', '2010', ' ', 'at', ' ', 'a', ' ', 'midwestern', ' ', 'medical', ' ', 'school', '. ', 'During', ' ', 'their', ' ', 'second', ' ', 'year', ', ', 'these', ' ', 'students', ' ', 'took', ' ', 'four', ' ', 'CPEs', ' 

... (1,591 outputs left out)

parts:  ['We', ' ', 'have', ' ', 'reported', ' ', 'previously', ' ', 'that', ' ', 'cerulein', '-', 'induced', ' ', 'edematous', ' ', 'pancreatitis', ' ', 'would', ' ', 'transform', ' ', 'into', ' ', 'hemorrhagic', ' ', 'pancreatitis', ' ', 'by', ' ', 'administration', ' ', 'of', ' ', 'endothelin', '-', '1', ' ', 'in', ' ', 'rats', '. ', 'In', ' ', 'the', ' ', 'present', ' ', 'study', ', ', 'we', ' ', 'tried', ' ', 'to', ' ', 'protect', ' ', 'rat', ' ', 'model', ' ', 'from', ' ', 'developing', ' ', 'into', ' ', 'hemorrhagic', ' ', 'pancreatitis', ' ', 'with', ' ', 'BQ123', ' (', 'an', ' ', 'ETA', ' ', 'receptor', ' ', 'antagonist', ').', '']
parts:  ['The', ' ', 'rat', ' ', 'model', ' ', 'was', ' ', 'made', ' ', 'by', ' ', '5', '-', 'hour', ' ', 'restraint', ' ', 'water', '-', 'immersion', ' ', 'stress', ' ', 'and', ' ', 'two', ' ', 'intraperitoneal', ' ', 'injections', ' ', 'of', ' ', 'cerulein', ' (', '40', ' ', 'micrograms', '/', 'kg', ') ', 'at', ' ', 'hourly', ' ', 'interval', '. '

Building dataset: 100%|█████████▉| 799/800 [1:24:30<00:05,  5.75s/it]

parts:  ['Recent', ' ', 'studies', ' ', 'have', ' ', 'found', ' ', 'a', ' ', 'choroidal', ' ', 'thickening', ' ', 'in', ' ', 'amblyopic', ' ', 'eyes', ' ', 'and', ' ', 'suggested', ' ', 'that', ' ', 'there', ' ', 'might', ' ', 'be', ' ', 'a', ' ', 'relationship', ' ', 'between', ' ', 'the', ' ', 'choroid', ' ', 'and', ' ', 'amblyopia', '. ', 'The', ' ', 'present', ' ', 'study', ' ', 'aimed', ' ', 'to', ' ', 'evaluate', ' ', 'the', ' ', 'effect', ' ', 'of', ' ', 'a', ' ', 'six', '-', 'month', ' ', 'treatment', ' ', 'of', ' ', 'amblyopia', ' ', 'on', ' ', 'choroidal', ' ', 'thickness', ' ', 'in', ' ', 'anisometropic', ' ', 'hyperopic', ' ', 'amblyopic', ' ', 'children', '.', '']
parts:  ['Thirty', '-', 'two', ' ', 'anisometropic', ' ', 'hyperopic', ' ', 'children', ' ', 'with', ' ', 'unilateral', ' ', 'amblyopia', ' ', 'were', ' ', 'included', ' ', 'in', ' ', 'this', ' ', 'prospective', ' ', 'study', '. ', 'Subfoveal', ' ', 'choroidal', ' ', 'thickness', ' ', 'was', ' ', 'measured', ' ',

Building dataset: 100%|██████████| 200/200 [00:00<00:00, 5894.39it/s]


In [ ]:
import torch
import transformers
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.1")


# tokenizer = AutoTokenizer.from_pretrained(
#     "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"
# )

train_encodings = tokenizer(
    train_questions,
    train_contexts,
    truncation=True,
    padding=True,
    max_length=512,
)
val_encodings = tokenizer(
    val_questions,
    val_contexts,
    truncation=True,
    padding=True,
    max_length=512,
)


In [ ]:
import torch

class QADataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = QADataset(train_encodings, train_labels)
val_dataset = QADataset(val_encodings, val_labels)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.1",
    num_labels=3
)


# model = AutoModelForSequenceClassification.from_pretrained(
#     "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract",
#     num_labels=3
# )

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne

In [ ]:
from transformers import Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro")
    }

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    weight_decay=0.01,
    warmup_steps=100,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
import pandas as pd
train_df = pd.DataFrame({
"question": train_questions,
"context": train_contexts,
"label": train_labels,
})

val_df = pd.DataFrame({
"question": val_questions,
"context": val_contexts,
"label": val_labels,
})

In [ ]:
train_df.to_csv("pubmedqa_aug_train.csv", index=False)
val_df.to_csv("pubmedqa_val.csv", index=False)

files.download("pubmedqa_aug_train.csv")
files.download("pubmedqa_val.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.401836,0.845000,0.820327
2,No log,0.535843,0.825000,0.773135
3,0.416383,0.607938,0.840000,0.783930


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=600, training_loss=0.36333667278289794, metrics={'train_runtime': 111.7095, 'train_samples_per_second': 42.969, 'train_steps_per_second': 5.371, 'total_flos': 1262944405094400.0, 'train_loss': 0.36333667278289794, 'epoch': 3.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.40197622776031494,
 'eval_accuracy': 0.84,
 'eval_f1': 0.8161693555372421,
 'eval_runtime': 1.4824,
 'eval_samples_per_second': 134.92,
 'eval_steps_per_second': 16.865,
 'epoch': 3.0}

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

print("Generating predictions...")

predictions = trainer.predict(main_test_data)

logits = predictions.predictions
y_pred = np.argmax(logits, axis=1)
y_true = predictions.label_ids

labels = ["yes", "no", "maybe"]

print("\nConfusion Matrix:")
cm = confusion_matrix(y_true, y_pred)
print(cm)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=labels))

Generating predictions...



Confusion Matrix:
[[96  2  5]
 [10 60  2]
 [ 2  7 16]]

Classification Report:
              precision    recall  f1-score   support

         yes       0.89      0.93      0.91       103
          no       0.87      0.83      0.85        72
       maybe       0.70      0.64      0.67        25

    accuracy                           0.86       200
   macro avg       0.82      0.80      0.81       200
weighted avg       0.86      0.86      0.86       200

